# Market-Making Simulator

## 1. Project Introduction

This notebook explores a simple **market-making simulator** built in Python. A market maker is a trader who continuously quotes both a **bid** (buy) and an **ask** (sell) price, earning the spread between them while managing the risk of holding inventory.

The goal of this project is to demonstrate, in a hands-on and quantitative way, several ideas that sit at the heart of trading:

- **Market microstructure** — how prices, quotes, and orders interact step by step
- **Expected value** — why quoting a spread is profitable on average
- **Inventory risk** — the danger of accumulating a large long or short position
- **Risk management** — measuring drawdown, volatility, and risk-adjusted return
- **PnL calculation** — marking a position to market over time

All the logic lives in the `src/` package (`market.py`, `trader.py`, `risk.py`, `simulation.py`, `visualisations.py`). This notebook ties it together and interprets the results.

In [ ]:
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.append(src_path)

import pandas as pd
import matplotlib.pyplot as plt

from simulation import run_market_making_simulation
from visualisations import plot_pnl_chart, plot_inventory_chart, plot_spread_analysis

## 2. What is Market Making?

A market maker provides **liquidity**: they stand ready to buy from sellers and sell to buyers at any moment. In exchange for taking on this service, they quote two prices around their estimate of the asset's fair value:

- **Bid price** = fair value - half the spread (the price they will *buy* at)
- **Ask price** = fair value + half the spread (the price they will *sell* at)

Every time a customer trades against them, the market maker captures roughly **half the spread** in expectation. If they buy at the bid and later sell at the ask, they pocket the full spread.

The catch is **inventory risk**. When more customers sell than buy, the maker accumulates a large long position — and if the fair value then drops, they lose money on that inventory. A good market maker therefore **skews their quotes** based on inventory: when long, they lower both quotes to encourage selling and discourage buying, nudging their position back toward flat.

## 3. Simulation Assumptions

This is a deliberately simple model. The key assumptions are:

- **Fair value** follows a random walk: each step it moves by a uniform random amount in `[-volatility, +volatility]`.
- **Customer orders** arrive with probability `order_probability` each step; when one arrives it is a buy or sell according to `buy_probability` / `sell_probability`.
- The market maker **always quotes** a symmetric spread around an inventory-adjusted fair value.
- Orders are **filled at the maker's quotes** (no slippage, no queue, no competition).
- Inventory is **capped** at `max_inventory`; orders that would breach the cap are refused.
- **PnL is marked to market**: `PnL = cash + inventory x fair_value`.

These simplifications keep the focus on the core intuition rather than on realistic exchange mechanics.

## 4. Run Base Simulation

We run a baseline scenario: 1000 time steps, a starting fair value of 100, unit volatility, balanced buy/sell flow, and a base spread of 2.0. The random seed is fixed at 42 so the run is fully reproducible.

In [ ]:
results, metrics = run_market_making_simulation(
    num_steps=1000,
    initial_fair_value=100.0,
    volatility=1.0,
    order_probability=0.7,
    buy_probability=0.5,
    sell_probability=0.5,
    trade_size=1,
    base_spread=2.0,
    max_inventory=20,
    inventory_penalty=0.1,
    random_seed=42
)

results.head()

The summary metrics for this run:

In [ ]:
metrics

Save the full step-by-step results to CSV so they can be tracked in the repository.

In [ ]:
results.to_csv("../results/simulation_results.csv", index=False)

## 5. Analyse PnL

The PnL chart shows how the market maker's mark-to-market profit evolves over the simulation. We expect a generally **upward drift** — the spread is captured on every trade — punctuated by **dips** whenever the maker is caught holding inventory as the fair value moves against them.

In [ ]:
plot_pnl_chart(results, "../results/pnl_chart.png")

## 6. Analyse Inventory

Inventory should oscillate around **zero**: the inventory-penalty term in the quote logic continually pulls the position back toward flat. Watch for the position pressing against the `+/-max_inventory` cap — those are the moments of greatest risk, where the maker is most exposed to an adverse price move.

In [ ]:
plot_inventory_chart(results, "../results/inventory_chart.png")

## 7. Spread Sensitivity Analysis

There is a fundamental trade-off in the choice of spread:

- A **wider spread** earns more per trade, but fewer customers cross it, so fewer trades happen.
- A **narrower spread** earns less per trade, but attracts far more volume.

This analysis re-runs the simulation across a range of base spreads and plots the resulting final PnL, revealing where the sweet spot lies for this particular market.

In [ ]:
plot_spread_analysis("../results/spread_analysis.png")

## 8. Risk Metrics

Profit alone does not tell the whole story — we also care about **how much risk** was taken to earn it. The metrics dictionary reports:

- **final_pnl** — total mark-to-market profit at the end of the run
- **max_drawdown** — the largest peak-to-trough fall in PnL (worst losing stretch)
- **pnl_volatility** — the standard deviation of step-to-step PnL changes
- **sharpe_ratio** — average PnL change per unit of volatility (risk-adjusted return)
- **max_absolute_inventory** — the largest position held in either direction
- **number_of_trades** — how much volume was transacted

In [ ]:
from IPython.display import Image, display

display(Image(filename="../results/pnl_chart.png"))
display(Image(filename="../results/inventory_chart.png"))
display(Image(filename="../results/spread_analysis.png"))

In [ ]:
for name, value in metrics.items():
    print(f"{name}: {value}")

## 9. Key Findings

- The market maker is **profitable on average**: capturing the spread across many trades produces a steadily rising PnL, exactly as expected value predicts.
- **Inventory control works**: skewing quotes by an inventory penalty keeps the position oscillating around zero and prevents runaway exposure.
- **Spread width is a trade-off**: too tight and each trade earns too little; too wide and volume collapses. The sensitivity analysis locates the profit-maximising region.
- **Risk is real**: drawdowns occur whenever the maker is holding inventory during an adverse price move, which is why risk-adjusted measures like the Sharpe ratio matter as much as raw PnL.

## 10. Limitations

This model is intentionally simplified, and a realistic market maker would need to account for much more:

- **No competition** — real makers compete with others for the same order flow, and quotes must stay competitive.
- **No adverse selection** — here order flow is random; in reality informed traders tend to trade against stale quotes.
- **Random-walk prices** — real prices have volatility clustering, jumps, and trends rather than uniform noise.
- **Perfect fills** — no slippage, latency, queue position, or partial fills are modelled.
- **Fixed parameters** — spread, inventory penalty, and limits are static, whereas a real strategy would adapt them dynamically to changing conditions.

Despite these simplifications, the simulator captures the essential economics of market making: **earn the spread, control inventory, and manage risk.**